In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

In [0]:
dbutils.widgets.text("masterPipelineRunId","")

run_id = dbutils.widgets.get("masterPipelineRunId")

print("Pipeline Run ID:", run_id)

In [0]:
masterPipelineRunID = '1234'

In [0]:
access_key = dbutils.secrets.get(scope='insurancescope',key='insurancesecret')

In [0]:
spark.conf.set("fs.azure.account.key.insurancestgacc2002.dfs.core.windows.net",access_key )

In [0]:
folderPath = "abfss://insurance@insurancestgacc2002.dfs.core.windows.net"

In [0]:
dbutils.fs.ls("abfss://insurance@insurancestgacc2002.dfs.core.windows.net")

In [0]:
dbutils.secrets.listScopes()

In [0]:
server = dbutils.secrets.get(scope="insurance-kv-scope", key="sql-server-name")
database = dbutils.secrets.get(scope="insurance-kv-scope", key="sql-db-name")
username = dbutils.secrets.get(scope="insurance-kv-scope", key="sql-user")
password = dbutils.secrets.get(scope="insurance-kv-scope", key="sql-password")

In [0]:
server = dbutils.secrets.get(scope="insurance-kv-scope", key="sql-server-name")
database = dbutils.secrets.get(scope="insurance-kv-scope", key="sql-db-name")
username = dbutils.secrets.get(scope="insurance-kv-scope", key="sql-user")
password = dbutils.secrets.get(scope="insurance-kv-scope", key="sql-password")

jdbc_url = f"jdbc:sqlserver://{server}:1433;databaseName={database}"

In [0]:
configDF = spark.read \
  .format("jdbc") \
  .option("url", f"jdbc:sqlserver://{server}:1433;databaseName={database}") \
  .option("dbtable", "insurance.api_config_table") \
  .option("user", username) \
  .option("password", password) \
  .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
  .load()

display(configDF)

In [0]:
def get_current_year():
    return datetime.now().year
year = get_current_year()

In [0]:
def flattenJson(df):
    while True:
        # Identify StructType and ArrayType columns
        complex_fields = [(field.name, field.dataType) for field in df.schema.fields 
                          if isinstance(field.dataType, (StructType, ArrayType))]

        # Stop when no more nested fields exist
        if not complex_fields:
            break

        for col_name, col_type in complex_fields:
            if isinstance(col_type, StructType):
                # Expand StructType fields into individual columns
                new_columns = [col(f"`{col_name}`.`{subfield.name}`").alias(f"{col_name}_{subfield.name}") 
                               for subfield in col_type.fields]
                df = df.select("*", *new_columns).drop(col_name)

            elif isinstance(col_type, ArrayType):
                # Explode ArrayType into multiple rows while preserving other columns
                df = df.withColumn(col_name, explode_outer(col(f"`{col_name}`"))).withColumn("RowSequence_"+ col_name, monotonically_increasing_id() + 1)
    
    # Rename columns to replace '.' with '_'
    df = df.toDF(*[col_name.replace(".", "_") for col_name in df.columns])
    
    return df


In [0]:
def load_data_into_delta(ObjectName,filePath):
    df = spark.read.format("json").option('multiline','true').load(filePath)
    df = flattenJson(df)
    df = df.withColumn("Added_By",lit(masterPipelineRunID))\
           .withColumn("Added_On",current_timestamp())\
           .withColumn("Modified_By",lit(masterPipelineRunID))\
           .withColumn("Modified_On",current_timestamp())
    df.write.mode("overwrite").saveAsTable(f'insurancetej.bronze.{ObjectName}')

In [0]:
for i in configDF.collect():
    print(i['Object_Name'])

In [0]:
try:
    for row in configDF.collect():
        ObjectName = row['Object_Name']
        filePath = row['Stage_Folder_Path']
        load_data_into_delta(ObjectName,folderPath+'/'+filePath+'/'+str(year)+'/*.json')
except Exception as e:
    print(f"Error occurred: {e}")
    raise

In [0]:
files = dbutils.fs.ls(f"abfss://insurance@insurancestgacc2002.dfs.core.windows.net/{ObjectName}/in/{year}")

json_files = [f for f in files if f.name.endswith(".json")]

if len(json_files) == 0:
    print("No files to process")
else:
    df = spark.read.option("multiline","true").json(f"abfss://insurance@insurancestgacc2002.dfs.core.windows.net/{ObjectName}/in/{year}/*.json")

In [0]:
#Move File to archive folder
try:
    for row in configDF.collect():
            ObjectName = row['Object_Name']
            sourcePath = f"abfss://insurance@insurancestgacc2002.dfs.core.windows.net/{ObjectName}/in"
            archivePath = f"abfss://insurance@insurancestgacc2002.dfs.core.windows.net/{ObjectName}/archive"
            for f in dbutils.fs.ls(sourcePath+"/"+str(year)):
                if f.name.endswith(".json"):
                    dbutils.fs.mv(f.path, archivePath + '/'+str(year)+'/'+f.name, True)
except Exception as e:
    print(f"Error occurred: {e}")
    raise
